In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import matplotlib.ticker as ticker
from matplotlib import rcParams
import matplotlib.colors as mcolors
import os
import warnings

# Ignore warnings
warnings.filterwarnings("ignore")

# ==========================================
# === 0. User-controllable parameters (Vertical line and text positions) ===
# ==========================================
ANNOT_POS_CONFIG = {
    0: {"offset_x": 0.5, "y_ratio": 0.85}, 
    2: {"offset_x": 0.5, "y_ratio": 0.85}, 
    4: {"offset_x": 0.5, "y_ratio": 0.85}, 
    8: {"offset_x": 0.5, "y_ratio": 0.85}, 
}
persist_hours = 0.0      # 0.0 = No persistence required (earliest intersection); change to 0.5 or 1.0 to avoid transient crossings

# ==========================================
# === 0.1 Global Size and Layout Control ===
# ==========================================
# Canvas size
FIG_WIDTH = 10         # Canvas width
FIG_HEIGHT = 7.5       # Canvas height

# Font sizes
FONT_TITLE = 30        # Main title font size
FONT_LABEL = 30        # X/Y axis label font size
FONT_TICK = 26         # X/Y axis tick number font size
FONT_OFFSET = 26       # Y axis top-left scientific notation font size
FONT_LEGEND = 18       # Legend font size
LEGEND_BBOX_X = 0.99   # Legend horizontal position
LEGEND_BBOX_Y = 1.015  # Legend vertical position
FONT_INSET_TICK = 24   # Inset plot tick number font size
FONT_INSET_OFFSET = 22 # Inset plot scientific notation font size
FONT_INSET_TEXT = 20   # Inset plot text font size

# Line and tick control
LINE_AXES = 2.0        
TICK_LENGTH_MAIN = 6.0 
TICK_LENGTH_INSET = 4.0
LINE_PLOT_MAIN = 2.2   
LINE_PLOT_INSET = 2.0  
LINE_VLINE = 1.8       
LINE_ZOOM_BOX = 1.5    

# Padding control
PAD_TITLE = 20         
PAD_YLABEL = 15        

# ==========================================
# === 1. Global Plot Style Settings ===
# ==========================================
config = {
    "font.family": 'sans-serif',
    "font.sans-serif": ['Arial'],
    "font.weight": "normal",
    "axes.labelweight": "normal",
    "axes.titleweight": "normal",
    "mathtext.fontset": 'custom',
    "mathtext.rm": 'Arial',
    "mathtext.it": 'Arial:italic',
    "mathtext.bf": 'Arial:bold',
    "axes.linewidth": LINE_AXES,
    "xtick.major.width": LINE_AXES,
    "ytick.major.width": LINE_AXES,
    "xtick.major.size": TICK_LENGTH_MAIN,
    "ytick.major.size": TICK_LENGTH_MAIN,
    "xtick.direction": 'out',
    "ytick.direction": 'out',
    "xtick.labelsize": FONT_TICK,
    "ytick.labelsize": FONT_TICK,
    "axes.labelsize": FONT_LABEL,
    "figure.figsize": (FIG_WIDTH, FIG_HEIGHT),
    "axes.unicode_minus": False
}
rcParams.update(config)

# ==============================================================================
# === 2. Core Module: Strict Stratified Logic to Find H ===
# ==============================================================================
target_min, target_max = 1.77, 2.63
i_min, i_max = 5, 12
j_min, j_max = 1, 2

S0_total = 1e7
R0_total = 1e6

S0_final, R0_final = None, None
num_S, num_R, H_final = 0, 0, 0

np.random.seed(42) 
while True:
    i = np.random.randint(i_min, i_max + 1)
    j = np.random.randint(j_min, j_max + 1)
    s_props = np.random.dirichlet(np.ones(i) * 0.5) 
    S_candidates = s_props * S0_total
    r_props = np.random.dirichlet(np.ones(j) * 0.5)
    R_candidates = r_props * R0_total
    all_pop = np.concatenate([S_candidates, R_candidates])
    total_pop = np.sum(all_pop) 
    p = all_pop / total_pop
    H_calc = -np.sum(p * np.log(p))
    if target_min <= H_calc <= target_max:
        S0_final, R0_final = S_candidates, R_candidates
        num_S, num_R, H_final = i, j, H_calc
        break

S0, R0 = S0_final, R0_final
np.random.seed(888)  

# === 3. Biological Parameters ===
K_cap, d, Vsmin, Vrmin = 1e9, 0.07, -0.1, -0.1
DELTA_MIN, DELTA_MAX, GAMMA_IB_CAP, EPS = 0.02, 0.12, 1.40, 1e-6

gamma_intra = np.random.uniform(0.95, 1.05)
gamma_ia = gamma_jb = gamma_intra
gamma_ja = np.random.uniform(0.80, 1.20)
delta = np.random.uniform(DELTA_MIN, DELTA_MAX)
gamma_ib = min(gamma_ja + delta, GAMMA_IB_CAP)
if gamma_ib <= gamma_ja: gamma_ib = gamma_ja + EPS

Vsmax = np.random.uniform(0.98, 1.0, num_S)
Vrmax = np.random.uniform(0.89, 0.90, num_R)
MICs = np.random.uniform(2.0, 8.0, num_S)
Ks = np.random.uniform(4.0, 6.0, num_S)
MICr = np.random.uniform(64.0, 256.0, num_R)
Kr = np.random.uniform(1.0, 2.0, num_R)

total_time = 72
c_values = [0, 2, 4, 8] 
t_span = np.linspace(0, total_time, 5000)

# === 4. Define ODE Equations ===
def bacteria_growth(y, t, c):
    S, R = y[:num_S], y[num_S:]
    total_S, total_R = np.sum(S), np.sum(R)
    comp_s = 1 - (gamma_ia * total_S + gamma_ja * total_R) / K_cap
    comp_r = 1 - (gamma_ib * total_S + gamma_jb * total_R) / K_cap
    Vs_rate = Vsmax - ((Vsmax - Vsmin) * (c / MICs) ** Ks) / ((c / MICs) ** Ks - (Vsmin / Vsmax))
    Vr_rate = Vrmax - ((Vrmax - Vrmin) * (c / MICr) ** Kr) / ((c / MICr) ** Kr - (Vrmin / Vrmax))
    return np.concatenate([Vs_rate * S * comp_s - d * S, Vr_rate * R * comp_r - d * R])

# === 5. Color Mapping ===
lancet_red = '#ED0000' 
colors_r = plt.cm.Reds(np.linspace(0.6, 1.0, num_R)) if num_R > 1 else [lancet_red]
colors_s = mcolors.LinearSegmentedColormap.from_list("LG", ['#ADB6C4', '#00468B'])(np.linspace(0.2, 1.0, num_S))

output_dir = "./results_dynamics"
if not os.path.exists(output_dir): os.makedirs(output_dir)

# === 7. Loop Simulation and Plotting ===
for c in c_values:
    y0 = np.concatenate([S0, R0])
    solution = odeint(bacteria_growth, y0, t_span, args=(c,))
    total_S_curve = np.sum(solution[:, :num_S], axis=1)
    total_R_curve = np.sum(solution[:, num_S:], axis=1)
    diff = total_R_curve - total_S_curve
    idx = np.where((diff[:-1] <= 0) & (diff[1:] > 0))[0] + 1
    t_cross, y_cross = None, None
    for i in idx:
        t_cross = t_span[i-1] - diff[i-1] * (t_span[i] - t_span[i-1]) / (diff[i] - diff[i-1])
        y_cross = total_R_curve[i-1] + (total_R_curve[i] - total_R_curve[i-1]) * (t_cross - t_span[i-1]) / (t_span[i] - t_span[i-1])
        break

    y_top_limit = min(K_cap * 1.2, max(100, np.max(solution) * 1.15))
    fig, ax = plt.subplots()
    
    # --- Main plot ---
    for k in range(num_S): ax.plot(t_span, solution[:, k], color=colors_s[k], linewidth=LINE_PLOT_MAIN, zorder=5)
    for k in range(num_R): ax.plot(t_span, solution[:, num_S + k], color=colors_r[k], linewidth=LINE_PLOT_MAIN, zorder=10)

    # Add invisible proxy lines for legends (using the darkest color at the end of the array)
    ax.plot([], [], color=colors_s[-1], linewidth=LINE_PLOT_MAIN, label='Sensitive')
    ax.plot([], [], color=colors_r[-1], linewidth=LINE_PLOT_MAIN, label='Resistant')

    ax.set_xlim(0, 72); ax.set_ylim(0, y_top_limit); ax.set_xticks(np.arange(0, 73, 12))
    ax.set_xlabel("Time (h)"); ax.set_ylabel("Species abundance", labelpad=PAD_YLABEL)
    ax.set_title(rf"$H$ = {H_final:.2f}, $c$ = {c}", fontsize=FONT_TITLE, pad=PAD_TITLE) 

    # Legend position and font size
    leg = ax.legend(loc='upper right', bbox_to_anchor=(LEGEND_BBOX_X, LEGEND_BBOX_Y), fontsize=FONT_LEGEND, frameon=False)
    leg.set_zorder(100)

    ax.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
    ax.yaxis.get_offset_text().set_fontsize(FONT_OFFSET)

    if t_cross is not None:
        axins = ax.inset_axes([0.58, 0.32, 0.38, 0.38], zorder=30)
        for k in range(num_S): axins.plot(t_span, solution[:, k], color=colors_s[k], linewidth=LINE_PLOT_INSET)
        for k in range(num_R): axins.plot(t_span, solution[:, num_S + k], color=colors_r[k], linewidth=LINE_PLOT_INSET)
        axins.set_xlim(0, max(12.0, np.ceil(t_cross) + 3.0)); axins.set_ylim(0, max(S0_total * 2.0, y_cross * 1.5))
        axins.axvline(x=t_cross, color='black', linestyle='--', linewidth=LINE_VLINE, zorder=20)
        axins.text(t_cross + 0.4, axins.get_ylim()[1] * 0.85, rf'$\hat{{t}} = {t_cross:.2f}$', family='Arial', fontsize=FONT_INSET_TEXT, fontweight='bold')
        axins.tick_params(axis='both', labelsize=FONT_INSET_TICK, length=TICK_LENGTH_INSET, width=LINE_AXES)
        ax.indicate_inset_zoom(axins, edgecolor="dimgray", alpha=0.6, linewidth=LINE_ZOOM_BOX, zorder=25)

    ax.spines['top'].set_visible(True); ax.spines['right'].set_visible(True)
    ax.tick_params(top=False, right=False, which='both')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'Lancet_H_{H_final:.2f}_C_{c}.png'), dpi=600, bbox_inches='tight')
    plt.close()